In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'IND': ['Jarace Walker', 'Ben Sheppard'], 'WAS': ['Jaden Hardy', 'Tre Johnson'], 'TOR': ['Sandro Mamukelashvili', 'Immanuel Quickley', 'Collin Murray-Boyles'], 'CHA': ['Coby White'], 'NOP': ['Yves Missi', 'Jordan Hawkins'], 'SAC': ['DeMar DeRozan'], 'GSW': ['Kristaps Porziņģis', 'LJ Cryer']}

Out Players:
{'MIN': ['Jaden McDaniels', 'Anthony Edwards'], 'IND': ['Pascal Siakam', 'T.J. McConnell', 'Andrew Nembhard', 'Aaron Nesmith'], 'CHI': ['Anfernee Simons', 'Matas Buzelis', 'Josh Giddey', 'Nick Richards'], 'WAS': ['Alex Sarr', 'Tristan Vukcevic', 'Trae Young', "D'Angelo Russell", 'Anthony Davis'], 'MIL': ['Myles Turner', 'Kyle Kuzma', 'Bobby Portis', 'Gary Trent', 'Giannis Antetokounmpo', 'Ryan Rollins', 'Kevin Porter'], 'BKN': ['Ziaire Williams', 'Terance Mann', 'Nic Claxton', 'Noah Clowney'], 'MIA': ['Nikola Jović', 'Terry Rozier'], 'TOR': ['Chucky Hepburn'], 'CHA': ['PJ Hall'], 'UTA': ['Isaiah Collier', 'Elijah Harkless', 'Lauri Markkanen', 'Keyonte George'],

### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
72,NaN,2025-26,1642449,Tolu Smith,Tolu,1610612765,DET,Detroit Pistons,22501144,2026-04-06T00:00:00,DET @ ORL,L,13.033333,3,5,0.600,0,0,0.000,2,4,0.5,1,2,3,1,0,0,0,2,3,2,8,8,13.1,0,0,12.0,1,13:02,1,109.9,124.0,124.0,89.4,92.0,92.0,20.5,32.0,32.0,0.100,0.00,12.5,0.077,0.154,0.115,0.0,0.0,0.600,0.592,0.219,0.210,99.29,92.07,76.73,92.07,0.120,25,3.0,5.0,42,85,0.494,10,30,0.333,13,19,0.684,8,27,35,32,21.0,10,7,9,24,19,107,-16.0,100.6,103.9,114.3,118.3,-13.7,-14.4,0.762,1.52,21.6,0.277,0.682,0.473,0.204,0.553,0.573,107.0,103.50,86.25,103,0.428,1610612753,ORL,Orlando Magic,41,81,0.506,11,26,0.423,30,40,0.750,6,29,35,28,15.0,16,9,7,19,24,123,16.0,114.3,118.3,100.6,103.9,13.7,14.4,0.683,1.87,19.2,0.318,0.723,0.527,0.144,0.574,0.624,107.0,103.50,86.25,104,0.572,NaN,PF,25.0
73,NaN,2025-26,1631204,Marcus Sasser,Marcus,1610612765,DET,Detroit Pistons,22501144,2026-04-06T00:00:00,DET @ ORL,L,18.238333,2,8,0.250,1,3,0.333,0,0,0.0,0,2,2,4,2,0,0,2,1,0,5,6,11.4,0,0,12.0,1,18:14,1,94.9,100.0,100.0,86.0,89.2,89.2,8.9,10.8,10.8,0.286,2.00,28.6,0.000,0.105,0.049,14.3,14.3,0.313,0.313,0.213,0.222,104.54,100.01,83.34,100.01,0.036,39,2.0,8.0,42,85,0.494,10,30,0.333,13,19,0.684,8,27,35,32,21.0,10,7,9,24,19,107,-16.0,100.6,103.9,114.3,118.3,-13.7,-14.4,0.762,1.52,21.6,0.277,0.682,0.473,0.204,0.553,0.573,107.0,103.50,86.25,103,0.428,1610612753,ORL,Orlando Magic,41,81,0.506,11,26,0.423,30,40,0.750,6,29,35,28,15.0,16,9,7,19,24,123,16.0,114.3,118.3,100.6,103.9,13.7,14.4,0.683,1.87,19.2,0.318,0.723,0.527,0.144,0.574,0.624,107.0,103.50,86.25,104,0.572,NaN,PG,25.0
74,NaN,2025-26,203501,Tim Hardaway Jr.,Tim,1610612743,DEN,Denver Nuggets,22501147,2026-04-06T00:00:00,DEN vs. POR,W,23.183333,1,7,0.143,1,5,0.200,0,0,0.0,1,0,1,1,1,1,1,1,4,0,3,-24,10.7,0,0,10.0,1,23:11,1,107.7,108.0,108.0,144.0,150.0,150.0,-36.4,-42.0,-42.0,0.056,1.00,11.1,0.042,0.000,0.021,11.1,11.1,0.214,0.214,0.138,0.140,107.99,105.59,87.99,105.59,-0.043,50,1.0,7.0,52,101,0.515,12,38,0.316,21,24,0.875,17,28,45,37,12.0,10,5,5,26,24,137,5.0,128.6,129.2,121.9,125.7,6.7,3.5,0.712,3.08,22.7,0.408,0.627,0.520,0.113,0.574,0.614,97.3,95.55,79.62,106,0.545,1610612757,POR,Portland Trail Blazers,42,89,0.472,25,52,0.481,23,28,0.821,11,28,39,29,18.0,8,5,5,24,26,132,-5.0,121.9,125.7,128.6,129.2,-6.7,-3.5,0.690,1.61,19.1,0.373,0.592,0.480,0.171,0.612,0.651,97.3,95.55,79.62,105,0.455,NaN,SG,33.0
64,NaN,2025-26,1628975,Jevon Cart

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260407_114339.json


,home_team,away_team,commence_time,bookmakers
0,Washington Wizards,Chicago Bulls,2026-04-07 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Indiana Pacers,Minnesota Timberwolves,2026-04-07 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Brooklyn Nets,Milwaukee Bucks,2026-04-07 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Toronto Raptors,Miami Heat,2026-04-07 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Boston Celtics,Charlotte Hornets,2026-04-08 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
BOOKMAKER = 'Underdog'
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-07 11:43:39
US latest pull: 2026-04-07 11:42:19


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Collin Sexton,Over,22.5,-137,2026-04-07,2026-04-07T18:42:54Z,2026-04-07 11:43:39
1,PrizePicks,player_points,Collin Sexton,Under,22.5,-137,2026-04-07,2026-04-07T18:42:54Z,2026-04-07 11:43:39
2,PrizePicks,player_points,Will Riley,Over,21.5,-137,2026-04-07,2026-04-07T18:42:54Z,2026-04-07 11:43:39
3,PrizePicks,player_points,Will Riley,Under,21.5,-137,2026-04-07,2026-04-07T18:42:54Z,2026-04-07 11:43:39
4,PrizePicks,player_points,Tre Jones,Over,19.5,-137,2026-04-07,2026-04-07T18:42:54Z,2026-04-07 11:43:39


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional indexer is out-of-bounds
[SKIP] Ethan Thompson: single positional indexer is out-of-bounds
[SKIP] A.J. Green: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Devin Carter: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds
[SKIP] Herb Jones: single positional indexer is out-of-bounds
[SKIP] Devin Carter: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional in

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90
0,Tre Jones,AST,17.52,26.21,32.68,0.1176,0.2245,0.3140,2.06,5.88,10.26
1,Bub Carrington,AST,19.80,27.83,35.14,0.1007,0.1626,0.2786,1.99,4.53,9.79
2,Collin Sexton,AST,18.63,27.20,32.71,0.0348,0.1392,0.2550,0.65,3.79,8.34
3,Ayo Dosunmu,AST,23.10,32.07,37.58,0.0782,0.1492,0.2384,1.81,4.79,8.96
4,Quenton Jackson,AST,13.29,22.44,31.18,0.0594,0.1377,0.2774,0.79,3.09,8.65
5,Bones Hyland,AST,11.83,18.81,26.17,0.0850,0.1423,0.2638,1.01,2.68,6.90
6,Jalen Slawson,AST,11.71,21.34,29.39,0.0739,0.1480,0.3201,0.87,3.16,9.41
7,Nolan Traore,AST,16.79,26.17,32.80,0.0839,0.1801,0.2980,1.41,4.71,9.78
8,Ben Saraf,AST,14.56,22.36,28.40,0.0651,0.1648,0.2880,0.95,3.69,8.18
9,Scottie Barnes,AST,26.77,30.67,38.59,0.1066,0.2326,0.3265,2.85,7.13,12.60


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
247,Davion Mitchell,PTS,8.5,25.94,7.59,0.545,0.455
37,Kevin Durant,AST,5.0,35.94,5.31,0.530,0.345
210,Bennedict Mathurin,PTS,16.5,27.14,15.75,0.559,0.441
134,Pelle Larsson,REB,3.5,28.74,3.47,0.634,0.366
214,John Collins,PTS,11.5,26.09,15.29,0.774,0.226
160,Drake Powell,PTS,11.5,25.51,6.05,0.186,0.814
121,LaMelo Ball,REB,4.5,31.36,4.58,0.596,0.404
38,Amen Thompson,AST,5.0,33.49,3.87,0.408,0.461
129,Kevin Durant,REB,5.5,35.94,6.13,0.624,0.376
197,Stephen Curry,PTS,24.5,29.81,20.55,0.433,0.567


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
216,Max Christie,PTS,10.5,26.84,11.67,0.658,0.342,PTS,Underdog,Los Angeles Clippers,11.5,238.0,115.2,18.0,97.22,28.0,100.0,-120.0,0.500,0.545,8.9,8.5,4.46,-1.6,-2.0,0.359,0.360,0.640,-28.00,17.33,0.2,0.4,0.40,0.52,28.32,3.26,0.14,0.05,8.00,6.0
80,Kyle Filipowski,REB,9.5,23.61,7.56,0.382,0.618,REB,Underdog,New Orleans Pelicans,10.5,242.5,117.4,22.0,100.99,11.0,-122.0,100.0,0.550,0.500,8.5,8.0,3.06,-1.0,-1.5,0.327,0.372,0.628,-32.31,25.60,0.6,0.3,0.40,0.22,27.24,5.34,0.26,0.06,8.40,5.0
57,Ayo Dosunmu,REB,4.5,32.07,4.09,0.547,0.453,REB,Underdog,Indiana Pacers,-12.5,233.0,118.3,27.0,101.66,8.0,-115.0,-110.0,0.535,0.524,7.3,6.5,3.47,2.8,2.0,-0.807,0.790,0.210,47.70,-59.91,0.8,0.8,0.53,0.25,33.15,2.99,0.19,0.04,3.33,3.0
109,Alperen Sengun,REB,8.5,30.82,9.98,0.692,0.308,REB,Underdog,Phoenix Suns,1.0,221.0,112.9,10.0,98.30,24.0,100.0,-103.0,0.500,0.507,8.5,8.5,3.31,0.0,0.0,0.000,0.500,0.500,0.00,-1.46,0.4,0.5,0.40,0.63,33.31,4.53,0.24,0.05,7.75,4.0
153,Donte DiVincenzo,PTS,11.5,27.37,12.17,0.642,0.358,PTS,Underdog,Indiana Pacers,-12.5,233.0,118.3,27.0,101.66,8.0,-110.0,-105.0,0.524,0.512,10.6,10.0,6.42,-0.9,-1.5,0.140,0.444,0.556,-15.24,8.55,0.4,0.4,0.33,0.49,30.42,3.45,0.17,0.04,15.00,3.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
9,Scottie Barnes,AST,7.0,30.67,7.13,0.529,0.375,AST,PrizePicks,Miami Heat,-1.5,240.5,113.4,11.0,104.40,1.0,-137.0,-137.0,0.578,0.578,9.9,10.0,2.92,2.9,3.0,-0.993,0.840,0.160,45.31,-72.32,0.8,0.8,0.60,0.27,30.50,4.91,0.22,0.02,6.4,5.0
19,Cody Williams,AST,4.0,29.57,2.78,0.285,0.573,AST,PrizePicks,New Orleans Pelicans,10.5,242.5,117.4,22.0,100.99,11.0,-137.0,-137.0,0.578,0.578,4.2,4.0,2.44,0.2,0.0,-0.082,0.533,0.467,-7.79,-19.21,0.4,0.4,0.47,0.08,33.22,4.94,0.21,0.07,1.8,5.0
156,Ousmane Dieng,PTS,18.5,27.39,11.95,0.200,0.800,PTS,PrizePicks,Brooklyn Nets,-2.0,220.0,118.1,25.0,97.57,27.0,100.0,-110.0,0.500,0.524,13.9,11.5,8.45,-4.6,-7.0,0.544,0.293,0.707,-41.40,34.97,0.2,0.1,0.13,0.04,30.34,6.74,0.23,0.07,12.0,1.0
176,LaMelo Ball,PTS,20.5,31.36,19.98,0.591,0.409,PTS,PrizePicks,Boston Celtics,4.5,221.0,111.7,4.0,95.49,30.0,105.0,-135.0,0.488,0.574,21.2,20.0,6.34,-0.3,-1.5,0.047,0.481,0.519,-1.40,-9.66,0.2,0.3,0.40,0.48,28.56,5.25,0.29,0.06,26.0,4.0
199,Maxime Raynaud,PTS,17.5,27.25,12.52,0.268,0.732,PTS,PrizePicks,Golden State Warriors,14.5,235.0,114.1,15.0,100.26,17.0,-112.0,-105.0,0.528,0.512,18.6,17.0,9.06,2.1,0.5,-0.232,0.592,0.408,12.06,-20.34,0.4,0.5,0.53,0.24,32.93,5.35,0.20,0.06,7.0,2.0


In [11]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
78,Kon Knueppel,REB,4.5,30.74,5.21,0.667,0.333,REB,PrizePicks,Boston Celtics,4.5,221.0,111.7,4.0,95.49,30.0,-114.0,-104.0,0.533,0.510,5.7,4.0,3.06,1.2,-0.5,-0.392,0.652,0.348,22.39,-31.74,0.2,0.4,0.40,0.56,29.09,5.83,0.21,0.02,4.00,2.0
161,Jericho Sims,PTS,8.5,25.83,6.01,0.373,0.627,PTS,Underdog,Brooklyn Nets,-2.0,220.0,118.1,25.0,97.57,27.0,-104.0,-103.0,0.510,0.507,6.8,6.0,3.58,-1.7,-2.5,0.475,0.317,0.683,-37.82,34.61,0.2,0.3,0.27,0.09,24.25,6.01,0.11,0.04,3.75,4.0
107,Rui Hachimura,REB,4.0,27.74,4.22,0.542,0.332,REB,PrizePicks,Oklahoma City Thunder,17.0,222.0,106.0,1.0,100.44,14.0,-137.0,-137.0,0.578,0.578,3.0,3.0,2.16,-1.0,-1.0,0.463,0.322,0.678,-44.30,17.29,0.4,0.2,0.20,0.39,22.96,6.78,0.14,0.05,4.60,5.0
57,Ayo Dosunmu,REB,4.5,32.07,4.09,0.547,0.453,REB,Underdog,Indiana Pacers,-12.5,233.0,118.3,27.0,101.66,8.0,-115.0,-110.0,0.535,0.524,7.3,6.5,3.47,2.8,2.0,-0.807,0.790,0.210,47.70,-59.91,0.8,0.8,0.53,0.25,33.15,2.99,0.19,0.04,3.33,3.0
39,Reed Sheppard,AST,3.5,30.17,4.05,0.640,0.360,AST,PrizePicks,Phoenix Suns,1.0,221.0,112.9,10.0,98.30,24.0,-123.0,104.0,0.552,0.490,5.1,4.5,3.63,1.6,1.0,-0.441,0.670,0.330,21.47,-32.68,0.4,0.6,0.40,0.29,29.43,5.42,0.19,0.04,3.40,5.0


### Get top EVs

In [12]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 163  |  Pairs: 1059  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [13]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 66  |  Pairs: 294  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
